# 10 — テンプレートマッチング

## 目標
- `cv2.matchTemplate` で類似度マップを計算
- `cv2.minMaxLoc` で最良位置を特定
- 複数のマッチング手法を比較

## 手法一覧
| 手法 | 最良 | 特徴 |
|------|------|------|
| TM_SQDIFF | min | 単純二乗差 |
| TM_SQDIFF_NORMED | min | 正規化二乗差 |
| TM_CCORR_NORMED | max | 正規化相互相関 |
| TM_CCOEFF_NORMED | max | 正規化相関係数 (推奨) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils import DATA_DIR, load_image, show_nb

src = load_image(DATA_DIR / 'lena.png')
h, w = src.shape[:2]

# 中央1/4をテンプレートとして切り出す
tmpl = src[h//4:h*3//4, w//4:w*3//4].copy()
th, tw = tmpl.shape[:2]
print(f'src: {w}x{h}, template: {tw}x{th}')
show_nb([('src', src), ('template', tmpl)], cols=2)

In [ ]:
gray_src  = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
gray_tmpl = cv2.cvtColor(tmpl, cv2.COLOR_BGR2GRAY)

methods = [
    ('TM_SQDIFF',        cv2.TM_SQDIFF,        True),
    ('TM_SQDIFF_NORMED', cv2.TM_SQDIFF_NORMED, True),
    ('TM_CCORR_NORMED',  cv2.TM_CCORR_NORMED,  False),
    ('TM_CCOEFF_NORMED', cv2.TM_CCOEFF_NORMED, False),
]

pairs = []
for name, method, use_min in methods:
    res = cv2.matchTemplate(gray_src, gray_tmpl, method)
    mn, mx, mn_loc, mx_loc = cv2.minMaxLoc(res)
    loc = mn_loc if use_min else mx_loc
    score = mn if use_min else mx
    vis = src.copy()
    cv2.rectangle(vis, loc, (loc[0]+tw, loc[1]+th), (0, 0, 255), 3)
    print(f'{name:22s}: score={score:.4f}, loc={loc}')
    pairs.append((f'{name}\n{score:.3f}', vis))

show_nb(pairs, cols=2)

In [ ]:
# TM_CCOEFF_NORMED の類似度マップを可視化
res = cv2.matchTemplate(gray_src, gray_tmpl, cv2.TM_CCOEFF_NORMED)
plt.figure(figsize=(6, 4))
plt.imshow(res, cmap='hot')
plt.colorbar(label='similarity')
plt.title('TM_CCOEFF_NORMED heatmap')
plt.show()